# ============================================================
# Samskara Public Health Ingestion - Phase 1
# API -> Remote Raw Storage -> Bronze -> Audit -> Checkpoint
# ============================================================

In [ ]:
import os
import json
import uuid
# import duckdb
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
# import samskara as sm
# import fsspec
# import polars as pl

# ----------------------------
# 1. Config
# ----------------------------

In [ ]:
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
RUN_DATE = RUN_TS.strftime("%Y-%m-%d")
RUN_TS_STR = RUN_TS.strftime("%Y%m%d_%H%M%S")

DATASET_NAME = "cdc_public_health"
SOURCE_SYSTEM = "cdc_socrata"

# Example CDC Socrata dataset endpoint
# Replace dataset_id later with the final dataset you choose
CDC_DATASET_ID = "9bhg-hcku"
API_URL = f"https://data.cdc.gov/resource/{CDC_DATASET_ID}.json"

LIMIT_ROWS = 5000
max_rows = sm.param("MAX_ROWS", 5000)
if not max_rows:
    max_rows = LIMIT_ROWS
    print("not yet populated by scheduler")
else:
    print("populated by scheduler")

REMOTE_RAW_BASE = "/public_health/raw"
# REMOTE_ARROWLAKE_DB = "s3://YOUR_BUCKET/public_health/arrowlake/public_health_arrowlake"

LOCAL_TMP_DIR = "/tmp/samskara_public_health"
Path(LOCAL_TMP_DIR).mkdir(parents=True, exist_ok=True)

RAW_FILE_NAME = f"{DATASET_NAME}_{RUN_TS_STR}_{RUN_ID}.json"
LOCAL_RAW_FILE = f"{LOCAL_TMP_DIR}/{RAW_FILE_NAME}"

RAW_REMOTE_PATH = (
    f"{REMOTE_RAW_BASE}/dataset={DATASET_NAME}/"
    f"ingestion_date={RUN_DATE}/run_id={RUN_ID}/{RAW_FILE_NAME}"
)

print("RUN_ID:", RUN_ID)
print("API_URL:", API_URL)
print("RAW_REMOTE_PATH:", RAW_REMOTE_PATH)
print(max_rows)

# ----------------------------
# 2. Fetch source data
# ----------------------------

In [ ]:
params = {
    "$limit": max_rows
}

response = requests.get(API_URL, params=params, timeout=60)
response.raise_for_status()

records = response.json()

with open(LOCAL_RAW_FILE, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

row_count = len(records)

print(f"Fetched rows: {row_count}")
print(f"Local raw file: {LOCAL_RAW_FILE}")

# ----------------------------
# 3. Upload raw file to remote storage
# Requires fsspec + s3fs configured in Samskara environment
# ----------------------------

In [ ]:
storage = sm.get_storage_connection("S3 CloudFlare R2 - Dev")

storage.upload(LOCAL_RAW_FILE, f"public_health/raw/dataset={DATASET_NAME}/ingestion_date={RUN_DATE}/run_id={RUN_ID}/{RAW_FILE_NAME}")
print("Uploaded raw file to:", storage.url(f"public_health/raw/.../{RAW_FILE_NAME}"))

# ----------------------------
# 4. Convert to dataframe
# ----------------------------

In [ ]:
df = pl.DataFrame(records)

# if df.is_empty():
#     print("No data received from source.")
# else:
#     df["_source_system"] = SOURCE_SYSTEM
#     df["_dataset_name"] = DATASET_NAME
#     df["_dataset_id"] = CDC_DATASET_ID
#     df["_run_id"] = RUN_ID
#     df["_raw_file_path"] = RAW_REMOTE_PATH
#     df["_ingestion_ts_utc"] = RUN_TS.isoformat()
#     df["_ingestion_date"] = RUN_DATE

if df.is_empty():
    print("No data received from source.")
else:
    # Polars requires using .with_columns() to add new columns
    df = df.with_columns(
        _source_system = pl.lit(SOURCE_SYSTEM),
        _dataset_name = pl.lit(DATASET_NAME),
        _dataset_id = pl.lit(CDC_DATASET_ID),
        _run_id = pl.lit(RUN_ID),
        _raw_file_path = pl.lit(RAW_REMOTE_PATH),
        _ingestion_ts_utc = pl.lit(RUN_TS.isoformat()),
        _ingestion_date = pl.lit(RUN_DATE)
    )
    
df.head()

# ----------------------------
# 5. Connect to ArrowLake database
# Adjust this depending on your Samskara ArrowLake connection name
# ----------------------------

In [ ]:
con = sm.get_database_connection("public_health_arrowlake")

# Optional, depending on your setup
# con.execute("INSTALL httpfs;")
# con.execute("LOAD httpfs;")

# If your remote storage uses S3-compatible config, set secrets/env vars in Samskara
# Example only:
# con.execute("SET s3_region='auto';")
# con.execute("SET s3_endpoint='YOUR_ENDPOINT';")
# con.execute("SET s3_access_key_id='YOUR_KEY';")
# con.execute("SET s3_secret_access_key='YOUR_SECRET';")
# con.execute("SET s3_url_style='path';")

print("Connected to ArrowLake DB")

# ----------------------------
# 6. Create metadata tables
# ----------------------------

In [ ]:
try:
  started_at = RUN_TS

  if df.is_empty():
    bronze_count = 0
  else:
    bronze_rows = [
        {
            "run_id": RUN_ID,
            "dataset_name": DATASET_NAME,
            "dataset_id": CDC_DATASET_ID,
            "source_system": SOURCE_SYSTEM,
            "raw_file_path": RAW_REMOTE_PATH,
            "raw_record_json": json.dumps(r),
            "ingestion_ts_utc": RUN_TS,
            "ingestion_date": RUN_DATE,
        }
        for r in records
    ]

    bronze_df = pd.DataFrame(bronze_rows)
    con.register("bronze_df", bronze_df)

    con.execute("""
    INSERT INTO public_health.bronze_public_health_events
    SELECT *
    FROM bronze_df
    """)

    bronze_count = con.execute("""
    SELECT COUNT(*)
    FROM public_health.bronze_public_health_events
    WHERE run_id = ?
    """, [RUN_ID]).fetchone()[0]

    completed_at = datetime.now(timezone.utc)

    con.execute("""
    INSERT INTO public_health.etl_ingestion_audit
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        RUN_ID,
        DATASET_NAME,
        SOURCE_SYSTEM,
        CDC_DATASET_ID,
        API_URL,
        RAW_REMOTE_PATH,
        row_count,
        bronze_count,
        "SUCCESS",
        None,
        started_at,
        completed_at,
        RUN_DATE
    ])

    con.execute("""
    DELETE FROM public_health.etl_file_checkpoint
    WHERE dataset_name = ? AND dataset_id = ?
    """, [DATASET_NAME, CDC_DATASET_ID])

    con.execute("""
    INSERT INTO public_health.etl_file_checkpoint
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, [
        DATASET_NAME,
        CDC_DATASET_ID,
        RUN_ID,
        RAW_REMOTE_PATH,
        completed_at,
        row_count,
        "SUCCESS"
    ])

    print("Bronze load successful")
    print("Bronze rows inserted:", bronze_count)

except Exception as e:
    completed_at = datetime.now(timezone.utc)

    con.execute("""
    INSERT INTO public_health.etl_ingestion_audit
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        RUN_ID,
        DATASET_NAME,
        SOURCE_SYSTEM,
        CDC_DATASET_ID,
        API_URL,
        RAW_REMOTE_PATH,
        row_count,
        0,
        "FAILED",
        str(e),
        RUN_TS,
        completed_at,
        RUN_DATE
    ])

    raise

# ----------------------------
# 7. Load bronze table
# ----------------------------

In [ ]:
# try:
#     started_at = RUN_TS

#     if df.is_empty():
#         bronze_count = 0
#     else:
#         con.register("incoming_df", df)

#         con.execute("""
#         CREATE TABLE IF NOT EXISTS bronze_public_health_events AS
#         SELECT * FROM incoming_df WHERE 1 = 0
#         """)

#         con.execute("""
#         INSERT INTO bronze_public_health_events
#         SELECT * FROM incoming_df
#         """)

#         bronze_count = con.execute("""
#         SELECT COUNT(*)
#         FROM bronze_public_health_events
#         WHERE _run_id = ?
#         """, [RUN_ID]).fetchone()[0]

#     completed_at = datetime.now(timezone.utc)

#     con.execute("""
#     INSERT INTO etl_ingestion_audit
#     VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
#     """, [
#         RUN_ID,
#         DATASET_NAME,
#         SOURCE_SYSTEM,
#         CDC_DATASET_ID,
#         API_URL,
#         RAW_REMOTE_PATH,
#         row_count,
#         bronze_count,
#         "SUCCESS",
#         None,
#         started_at,
#         completed_at,
#         RUN_DATE
#     ])

#     con.execute("""
#     DELETE FROM etl_file_checkpoint
#     WHERE dataset_name = ? AND dataset_id = ?
#     """, [DATASET_NAME, CDC_DATASET_ID])

#     con.execute("""
#     INSERT INTO etl_file_checkpoint
#     VALUES (?, ?, ?, ?, ?, ?, ?)
#     """, [
#         DATASET_NAME,
#         CDC_DATASET_ID,
#         RUN_ID,
#         RAW_REMOTE_PATH,
#         completed_at,
#         row_count,
#         "SUCCESS"
#     ])

#     print("Bronze load successful")
#     print("Bronze rows inserted:", bronze_count)

# except Exception as e:
#     completed_at = datetime.now(timezone.utc)

#     con.execute("""
#     INSERT INTO etl_ingestion_audit
#     VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
#     """, [
#         RUN_ID,
#         DATASET_NAME,
#         SOURCE_SYSTEM,
#         CDC_DATASET_ID,
#         API_URL,
#         RAW_REMOTE_PATH,
#         row_count,
#         0,
#         "FAILED",
#         str(e),
#         RUN_TS,
#         completed_at,
#         RUN_DATE
#     ])

#     raise

# ----------------------------
# 8. Validation queries
# ----------------------------

In [ ]:
con.execute("""
SELECT
    ingestion_date,
    dataset_name,
    COUNT(*) AS rows_loaded,
    COUNT(DISTINCT run_id) AS run_count
FROM public_health.bronze_public_health_events
GROUP BY ingestion_date, dataset_name
ORDER BY ingestion_date DESC
""").df()

In [ ]:
con.execute("""
SELECT *
FROM public_health.etl_ingestion_audit
ORDER BY started_at_utc DESC
LIMIT 10
""").df()